Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [ ]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

## Agent Architectures

### Install libraries

In [9]:
# (setup cell already installs what this notebook needs)

Note: you may need to restart the kernel to use updated packages.


### Import libraries and load configuration

In [10]:
import operator
from typing import Annotated, List, TypedDict, Literal
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()

True

### Sequential Pattern
The sequential pattern is the simplest multi-agent organization: a fixed pipeline where stages run one after another in a hard-coded order. Each stage takes the previous stage's output as its input and writes its own result into the shared state, like an assembly line. There's no routing decision and no loop : stage 1 always runs, then stage 2, then stage 3, every time. That makes it predictable and easy to debug, but rigid: it can't adapt, skip a step, or recover, and a weak result early on quietly degrades everything downstream. It fits workflows with fixed stages, extract → transform → summarize, or analyze → answer → evaluate, where the order is known in advance and never needs to change.

In [12]:
from typing_extensions import TypedDict
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

llm = make_llm()

class State(TypedDict):
    message: str        # the customer message (input)
    category: str       # stage 1 (LLM) writes this
    priority: str       # stage 2 (deterministic) writes this ; depends on category
    reply: str          # stage 3 (LLM) writes this ; prompt depends on category


# Stage 1 : LLM classifies the message into one fixed label
def classify(state: State) -> dict:
    prompt = (
        "Classify the customer message as exactly one word: "
        "complaint, question, or praise. Reply with only that word.\n\n"
        f"Message: {state['message']}"
    )
    raw = llm.invoke(prompt).content.strip().lower()
    # normalise to a known label (LLMs don't always return exactly one word)
    category = next((c for c in ("complaint", "question", "praise") if c in raw), "question")
    return {"category": category}


# Stage 2 : DETERMINISTIC branch: priority is chosen from stage 1's label
def prioritize(state: State) -> dict:
    priority = {"complaint": "HIGH", "question": "NORMAL", "praise": "LOW"}[state["category"]]
    return {"priority": priority}


# Stage 3 : LLM again, but the INSTRUCTION it gets depends on the category
def respond(state: State) -> dict:
    instructions = {
        "complaint": "Write a short, apologetic reply and promise a follow-up.",
        "question":  "Write a short, helpful reply that answers the question.",
        "praise":    "Write a short, warm reply thanking the customer.",
    }[state["category"]]
    prompt = f"{instructions}\n\nCustomer message: {state['message']}"
    return {"reply": llm.invoke(prompt).content.strip()}


builder = StateGraph(State)
builder.add_node("classify", classify)
builder.add_node("prioritize", prioritize)
builder.add_node("respond", respond)
builder.add_edge(START, "classify")
builder.add_edge("classify", "prioritize")
builder.add_edge("prioritize", "respond")
builder.add_edge("respond", END)
graph = builder.compile()


# Run on two different inputs to watch the pipeline diverge
for message in [
    "My order arrived broken and nobody has answered my emails!",
    "Do you ship to the Netherlands?",
]:
    result = graph.invoke({"message": message})
    print(f"message:  {message}")
    print(f"category: {result['category']}")
    print(f"priority: {result['priority']}")
    print(f"reply:    {result['reply']}\n")

message:  My order arrived broken and nobody has answered my emails!
category: complaint
priority: HIGH
reply:    Dear [Customer's Name],

I sincerely apologize for the inconvenience you've experienced with your order and for the lack of response to your emails. This is not the level of service we strive to provide. I assure you that I will look into this matter immediately and follow up with you shortly to resolve the issue.

Thank you for your patience.

Best regards,  
[Your Name]  
[Your Position]  
[Your Company]

message:  Do you ship to the Netherlands?
category: question
priority: NORMAL
reply:    Yes, we do ship to the Netherlands! If you have any specific questions about shipping options or delivery times, feel free to ask.



### Router Pattern
The router pattern has one defining move: a single routing decision sends the task to exactly one specialist, and that specialist handles it. Unlike the sequential pipeline (every stage runs, in order) or the supervisor (control comes back to the hub after each step), a router fires once and forwards : the dispatcher classifies the call, connects us to one expert, and steps out.
In LangGraph that "route to one of several" is exactly what a conditional edge expresses, so this pattern reuses the add_conditional_edges mechanism from early in our thread, now routing between specialists instead of between tool/no-tool. Here's a clear, minimal version with three specialist "agents" (kept as simple functions so the routing is the visible part):

In [14]:
from typing_extensions import TypedDict
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

llm = make_llm()

class State(TypedDict):
    question: str       # the incoming task
    category: str       # the router's decision
    answer: str         # the chosen specialist's output


# --- The router: classify the task, don't answer it ---
def router(state: State) -> dict:
    prompt = (
        "Classify the question into exactly one word: "
        "math, code, or general. Reply with only that word.\n\n"
        f"Question: {state['question']}"
    )
    raw = llm.invoke(prompt).content.strip().lower()
    category = next((c for c in ("math", "code", "general") if c in raw), "general")
    return {"category": category}


# --- The routing function: reads the decision, names the next node ---
def route(state: State) -> str:
    return state["category"]        # returns "math", "code", or "general"


# --- Three specialists, each with its own focused prompt ---
def math_agent(state: State) -> dict:
    p = f"You are a math tutor. Solve step by step:\n{state['question']}"
    return {"answer": llm.invoke(p).content.strip()}

def code_agent(state: State) -> dict:
    p = f"You are a programming expert. Answer with code:\n{state['question']}"
    return {"answer": llm.invoke(p).content.strip()}

def general_agent(state: State) -> dict:
    p = f"You are a helpful assistant. Answer clearly:\n{state['question']}"
    return {"answer": llm.invoke(p).content.strip()}


# --- Wire it: router -> ONE specialist -> END ---
builder = StateGraph(State)
builder.add_node("router", router)
builder.add_node("math", math_agent)
builder.add_node("code", code_agent)
builder.add_node("general", general_agent)

builder.add_edge(START, "router")
builder.add_conditional_edges(
    "router",
    route,                                        # decides where to go
    {"math": "math", "code": "code", "general": "general"},   # label -> node
)
builder.add_edge("math", END)      # each specialist goes straight to END
builder.add_edge("code", END)
builder.add_edge("general", END)

graph = builder.compile()


# --- Run: different questions reach different specialists ---
for q in ["What is 15% of 240?",
          "Write a Python function to reverse a string.",
          "Why is the sky blue?"]:
    result = graph.invoke({"question": q})
    print(f"[{result['category']:7}] {q}\n  -> {result['answer'][:70]}...\n")

[math   ] What is 15% of 240?
  -> To find 15% of 240, you can follow these steps:

1. **Convert the perc...

[code   ] Write a Python function to reverse a string.
  -> Certainly! Here is a simple Python function that reverses a string:

`...

[general] Why is the sky blue?
  -> The sky appears blue due to a phenomenon called Rayleigh scattering. W...



### Supervisor Pattern 
The supervisor pattern is a hierarchical architecture: a central orchestrator, the supervisor, coordinates a team of specialist worker agents. It receives the task, decides which worker should handle the next step, delegates to it, gets control back when that worker finishes, and decides again, dispatching to the same or a different worker as many times as needed, until the task is complete. The defining rule is that the supervisor controls all communication and delegation, choosing which agent to invoke based on the current context. It does no work itself; it only routes. Workers don't talk to each other or the user directly ; everything flows through the hub.

In [17]:
%pip install langgraph_supervisor

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\languages\Python311\python.exe -m pip install --upgrade pip


In [22]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph_supervisor import create_supervisor

model = make_llm()

# --- Worker tools ---
@tool
def web_search(query: str) -> str:
    """Search the web for information."""
    print("WEB_SEARCH CALLED")
    return ("FAANG 2024 headcounts: Meta 67,317; Apple 164,000; "
            "Amazon 1,551,000; Netflix 14,000; Alphabet 181,269.")

@tool
def add(a: float, b: float) -> float:
    """Add two numbers."""
    print("ADD CALLED")
    return a + b

# --- Two specialist workers, each with a name the supervisor delegates to ---
research_agent = create_react_agent(
    model, tools=[web_search], name="research_expert",
    prompt="You are a research expert. Find facts with web_search. Do no math.",
)
math_agent = create_react_agent(
    model, tools=[add], name="math_expert",
    prompt="You are a math expert. Use the add tool, one pair at a time.",
)

# --- Supervisor: coordinates the workers, does no work itself ---
supervisor = create_supervisor(
    [research_agent, math_agent],
    model=model,
    prompt=(
       "You manage two experts:\n"
        "- research_expert: finds facts via web search\n"
        "- math_expert: does arithmetic\n"
        "Delegate to one expert at a time. Do not do any work yourself. "
        "When the task is complete, give the final answer and stop."
    ),
).compile()

result = supervisor.invoke(
    {"messages": [{"role": "user",
                   "content": "What is the combined headcount of the FAANG companies in 2024? Use tools to calculate that. Do not use your mathematical knowledge"}]}
)
for m in result["messages"]:
    m.pretty_print()

C:\Users\wille\AppData\Local\Temp\ipykernel_13908\1495281302.py:23: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  research_agent = create_react_agent(
C:\Users\wille\AppData\Local\Temp\ipykernel_13908\1495281302.py:27: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  math_agent = create_react_agent(


WEB_SEARCH CALLED
WEB_SEARCH CALLED
WEB_SEARCH CALLED
WEB_SEARCH CALLED
WEB_SEARCH CALLED
WEB_SEARCH CALLED
ADD CALLED
ADD CALLED
ADD CALLED
ADD CALLED
ADD CALLED
================================ Human Message =================================

What is the combined headcount of the FAANG companies in 2024? Use tools to calculate that. Do not use your mathematical knowledge
================================== Ai Message ==================================
Name: supervisor
Tool Calls:
  transfer_to_research_expert (call_cNMxxYbtfCN0rlmdPkYo96CG)
 Call ID: call_cNMxxYbtfCN0rlmdPkYo96CG
  Args:
================================= Tool Message =================================
Name: transfer_to_research_expert

Successfully transferred to research_expert
================================== Ai Message ==================================
Name: research_expert

The combined headcount of the FAANG companies in 2024 is as follows:

- Meta (Facebook): 67,317
- Apple: 164,000
- Amazon: 1,551,000
- Netfl

### Swarm Pattern 
The swarm pattern is a decentralized multi-agent architecture in which specialist agents hand control directly to one another, with no central coordinator. Instead of a supervisor deciding every step, each agent decides for itself when its part is done and which peer should take over next. Control transfers happen through handoff tools: each agent is given a tool that, when called, passes control, and the shared message history, to a named peer. The system tracks an "active agent" in its state, so whichever agent last received control handles the next turn. Because that active agent is remembered, a follow-up message resumes with the same specialist rather than restarting from a fixed entry point. This makes the swarm well suited to multi-turn, conversational assistants where a user should stay with one expert until they need another. A checkpointer is required for this persistence ; without it, the swarm forgets who was active between turns. The routing is explicit rather than emergent: each handoff tool names exactly which peer can take over, so the possible transfers are predictable and bounded. 

In [24]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph_swarm import create_handoff_tool, create_swarm

llm = make_llm()

@tool
def search_flights(origin: str, destination: str) -> str:
    """Search for flights between two cities."""
    return f"Found 3 flights from {origin} to {destination}, cheapest €149."

@tool
def search_hotels(city: str) -> str:
    """Search for hotels in a city."""
    return f"Found 5 hotels in {city}, from €80/night."

# Each agent carries a handoff tool pointing at the other agent by name
flight_agent = create_agent(
    llm,
    tools=[search_flights, create_handoff_tool(
        agent_name="hotel_agent",
        description="Transfer to the hotel agent for accommodation questions.")],
    system_prompt=("You are a flight specialist. Handle flights only. "
                   "For hotels, hand off to hotel_agent."),
    name="flight_agent",           # <-- name is required; handoffs target it
)

hotel_agent = create_agent(
    llm,
    tools=[search_hotels, create_handoff_tool(
        agent_name="flight_agent",
        description="Transfer to the flight agent for flight questions.")],
    system_prompt=("You are a hotel specialist. Handle hotels only. "
                   "For flights, hand off to flight_agent."),
    name="hotel_agent",
)

# Checkpointer REQUIRED so the swarm remembers the active agent across turns
swarm = create_swarm(
    [flight_agent, hotel_agent],
    default_active_agent="flight_agent",
).compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "trip-1"}}

# Turn 1 : starts on the flight agent (the default active agent)
print(swarm.invoke(
    {"messages": [{"role": "user", "content": "Find a flight from Amsterdam to Rome."}]},
    config)["messages"][-1].content)

# Turn 2 : same thread. Asking about hotels makes flight_agent hand off to
# hotel_agent, which then stays active for the rest of the conversation.
print(swarm.invoke(
    {"messages": [{"role": "user", "content": "Now find me a hotel in Rome."}]},
    config)["messages"][-1].content)
for m in result["messages"]:
    m.pretty_print()

I found 3 flights from Amsterdam to Rome, with the cheapest ticket priced at €149. If you need more details or want to book a flight, let me know!
I found 5 hotels in Rome, with prices starting from €80 per night. If you need more information about the hotels or want to make a reservation, just let me know!
================================ Human Message =================================

What is the combined headcount of the FAANG companies in 2024? Use tools to calculate that. Do not use your mathematical knowledge
================================== Ai Message ==================================
Name: supervisor
Tool Calls:
  transfer_to_research_expert (call_cNMxxYbtfCN0rlmdPkYo96CG)
 Call ID: call_cNMxxYbtfCN0rlmdPkYo96CG
  Args:
================================= Tool Message =================================
Name: transfer_to_research_expert

Successfully transferred to research_expert
================================== Ai Message ==================================
Name: research_ex

### Sequential Agent

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    raw: str            # the input, e.g. "40, 45, 8, 30"
    numbers: list       # stage 1 writes this
    summary: str        # stage 2 writes this : depends on `numbers`
    action: str         # stage 3 writes this : depends on `summary`


# Stage 1: parse the raw string into numbers
def parse(state: State) -> dict:
    numbers = [int(x) for x in state["raw"].split(",")]
    return {"numbers": numbers}


# Stage 2: BEHAVES DIFFERENTLY depending on stage 1's numbers
def summarize(state: State) -> dict:
    total = sum(state["numbers"])
    if total > 100:
        return {"summary": f"large dataset (total {total})"}
    else:
        return {"summary": f"small dataset (total {total})"}


# Stage 3: BEHAVES DIFFERENTLY depending on stage 2's summary
def decide(state: State) -> dict:
    if "large" in state["summary"]:
        return {"action": "Route to the detailed-review queue."}
    else:
        return {"action": "Auto-approve : no review needed."}


builder = StateGraph(State)
builder.add_node("parse", parse)
builder.add_node("summarize", summarize)
builder.add_node("decide", decide)
builder.add_edge(START, "parse")
builder.add_edge("parse", "summarize")
builder.add_edge("summarize", "decide")
builder.add_edge("decide", END)
graph = builder.compile()


# Run on two DIFFERENT inputs to watch stages 2 and 3 diverge
for raw in ["40, 45, 8, 30", "3, 5, 2, 4"]:
    result = graph.invoke({"raw": raw})
    print(f"input:   {raw}")
    print(f"numbers: {result['numbers']}")
    print(f"summary: {result['summary']}")
    print(f"action:  {result['action']}\n")

### Sequential Agent Ollama

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    raw: str            # the input, e.g. "40, 45, 8, 30"
    numbers: list       # stage 1 writes this
    summary: str        # stage 2 writes this : depends on `numbers`
    action: str         # stage 3 writes this : depends on `summary`


# Stage 1: parse the raw string into numbers
def parse(state: State) -> dict:
    numbers = [int(x) for x in state["raw"].split(",")]
    return {"numbers": numbers}


# Stage 2: BEHAVES DIFFERENTLY depending on stage 1's numbers
def summarize(state: State) -> dict:
    total = sum(state["numbers"])
    if total > 100:
        return {"summary": f"large dataset (total {total})"}
    else:
        return {"summary": f"small dataset (total {total})"}


# Stage 3: BEHAVES DIFFERENTLY depending on stage 2's summary
def decide(state: State) -> dict:
    if "large" in state["summary"]:
        return {"action": "Route to the detailed-review queue."}
    else:
        return {"action": "Auto-approve : no review needed."}


builder = StateGraph(State)
builder.add_node("parse", parse)
builder.add_node("summarize", summarize)
builder.add_node("decide", decide)
builder.add_edge(START, "parse")
builder.add_edge("parse", "summarize")
builder.add_edge("summarize", "decide")
builder.add_edge("decide", END)
graph = builder.compile()


# Run on two DIFFERENT inputs to watch stages 2 and 3 diverge
for raw in ["40, 45, 8, 30", "3, 5, 2, 4"]:
    result = graph.invoke({"raw": raw})
    print(f"input:   {raw}")
    print(f"numbers: {result['numbers']}")
    print(f"summary: {result['summary']}")
    print(f"action:  {result['action']}\n")

### Sequential Agent 

In [9]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

llm = make_llm()


# --- Shared state: each stage reads earlier fields, writes its own ---
class State(TypedDict):
    data: str          # the raw input
    analysis: str      # produced by stage 1
    answer: str        # produced by stage 2
    evaluation: str    # produced by stage 3


# --- Three specialists, each a focused prompt ---
def analyze(state: State) -> dict:
    prompt = f"Analyze this data and list the key facts:\n{state['data']}"
    return {"analysis": llm.invoke(prompt).content}

def generate(state: State) -> dict:
    prompt = f"Using this analysis, write a clear answer:\n{state['analysis']}"
    return {"answer": llm.invoke(prompt).content}

def evaluate(state: State) -> dict:
    prompt = (f"Rate this answer for accuracy and clarity (1-10) with one line of "
              f"reasoning.\n\nAnswer:\n{state['answer']}")
    return {"evaluation": llm.invoke(prompt).content}


# --- Wire them in a fixed line: analyze -> generate -> evaluate ---
builder = StateGraph(State)
builder.add_node("analyze", analyze)
builder.add_node("generate", generate)
builder.add_node("evaluate", evaluate)

builder.add_edge(START, "analyze")
builder.add_edge("analyze", "generate")   # plain edges : no conditions, no loop
builder.add_edge("generate", "evaluate")
builder.add_edge("evaluate", END)

graph = builder.compile()


# --- Run ---
result = graph.invoke({"data": "Sales rose 12% in Q3, driven mainly by the EU region."})
print("ANALYSIS:  ", result["analysis"])
print("ANSWER:    ", result["answer"])
print("EVALUATION:", result["evaluation"])

ANALYSIS:   Here are the key facts from the provided data:

1. **Sales Growth**: There was a 12% increase in sales during the third quarter (Q3).
2. **Primary Driver**: The growth in sales was primarily driven by the performance in the European Union (EU) region.
3. **Time Frame**: The analysis pertains specifically to the third quarter of the fiscal year.

These points highlight the significant sales increase and its geographical source.
ANSWER:     In the third quarter of the fiscal year, sales experienced a notable increase of 12%. This growth was primarily driven by strong performance in the European Union (EU) region, highlighting the significance of this market in contributing to overall sales success during this period.
EVALUATION: Rating: 9

Reasoning: The answer is clear and accurately conveys the key information about sales growth and its regional impact, though it could benefit from more specific data or context regarding the overall sales figures.


### Sequential Agent
Sequential Agent performs internal reasoning in several steps (so-called scratchpad) but does not use tools; it is used where pure analysis and deduction are sufficient.

In [11]:
llm = make_llm()

class State(TypedDict):
    question: str
    steps: Annotated[List[str], operator.add]
    answer: str

def plan_node(state: State) -> dict:
    sys = (
        "You are a careful planner. Break the user's question into 2-4 concise steps. "
        "Do not solve. Return only a numbered list of steps; no extra text."
    )
    messages = [("system", sys), ("user", state["question"])]
    resp = llm.invoke(messages)
    raw = resp.content
    steps = []
    for line in str(raw).splitlines():
        line = line.strip()
        if not line:
            continue
        line = line.lstrip("-• ").split(". ", 1)[-1] if ". " in line[:4] else line.lstrip("-• ")
        steps.append(line)
    return {"steps": steps}

def solve_node(state: State) -> dict:
    """Use the planned steps to derive the final answer only."""
    sys = (
        "Use the provided steps to solve the problem. "
        "Return only the final answer, no reasoning."
    )
    messages = [
        ("system", sys),
        ("user", f"Question: {state['question']}\\\\nSteps: {state['steps']}"),
    ]
    resp = llm.invoke(messages)
    return {"answer": str(resp.content).strip()}

#  Wire up the graph
graph = StateGraph(State)
graph.add_node("plan", plan_node)
graph.add_node("solve", solve_node)

graph.add_edge(START, "plan")
graph.add_edge("plan", "solve")
graph.add_edge("solve", END)

cot_graph = graph.compile()

In [12]:
state = {
    "question": "If a book has 350 pages and I read 14 pages per day, how many days to finish?",
    "steps": [],
    "answer": ""
}
out = cot_graph.invoke(state)
print("Final answer:", out["answer"])

Final answer: 25 days


### Custom Agent
Custom Agent provides complete flexibility. We define the logic, routing, and nodes ourselves.

In [13]:
class CustomState(TypedDict):
    input: str
    task: Literal["math", "capitalize", "count"]
    result: str

def route(state: CustomState) -> str:
    """Deterministic router based on a simple protocol in the input."""
    text = state["input"].strip().lower()
    if text.startswith("math:"):
        return "math"
    if text.startswith("capitalize:"):
        return "capitalize"
    if text.startswith("count:"):
        return "count"
    return "count"

def do_math(state: CustomState) -> dict:
    expr = state["input"].split(":", 1)[-1].strip()
    allowed = set("0123456789+-*/(). ")
    if any(c not in allowed for c in expr):
        return {"result": "Error: unsupported characters in math expression."}
    try:
        res = eval(expr, {"__builtins__": {}})
    except Exception as e:
        res = f"Error: {e}"
    return {"result": str(res)}

def do_capitalize(state: CustomState) -> dict:
    text = state["input"].split(":", 1)[-1].strip()
    return {"result": text.upper()}

def do_count(state: CustomState) -> dict:
    text = state["input"].split(":", 1)[-1].strip()
    tokens = [t for t in text.split() if t]
    return {"result": f"words={len(tokens)} chars={len(text)}"}

graph = StateGraph(CustomState)
graph.add_node("math", do_math)
graph.add_node("capitalize", do_capitalize)
graph.add_node("count", do_count)

graph.add_conditional_edges(
    START,
    route,
    {
        "math": "math",
        "capitalize": "capitalize",
        "count": "count",
    },
)
graph.add_edge("math", END)
graph.add_edge("capitalize", END)
graph.add_edge("count", END)

custom_agent = graph.compile(debug=True)

In [14]:
for user_input in [
    "math: (12 + 8) * 3",
    "capitalize: langgraph is great!",
    "count: How many words are here?",
]:
    out = custom_agent.invoke({"input": user_input, "task": "count", "result": ""})
    print(f"Input: {user_input}\\nResult: {out['result']}\\n---")

[values] {'input': 'math: (12 + 8) * 3', 'task': 'count', 'result': ''}
[updates] {'math': {'result': '60'}}
[values] {'input': 'math: (12 + 8) * 3', 'task': 'count', 'result': '60'}
Input: math: (12 + 8) * 3\nResult: 60\n---
[values] {'input': 'capitalize: langgraph is great!', 'task': 'count', 'result': ''}
[updates] {'capitalize': {'result': 'LANGGRAPH IS GREAT!'}}
[values] {'input': 'capitalize: langgraph is great!', 'task': 'count', 'result': 'LANGGRAPH IS GREAT!'}
Input: capitalize: langgraph is great!\nResult: LANGGRAPH IS GREAT!\n---
[values] {'input': 'count: How many words are here?', 'task': 'count', 'result': ''}
[updates] {'count': {'result': 'words=5 chars=24'}}
[values] {'input': 'count: How many words are here?', 'task': 'count', 'result': 'words=5 chars=24'}
Input: count: How many words are here?\nResult: words=5 chars=24\n---


### Supervisor

In [32]:
class SupervisorState(TypedDict):
    """State for supervisor pattern with multiple agents."""
    topic: str
    messages: Annotated[List[str], operator.add]
    next_agent: str
    final_answer: str


def researcher_agent(state: SupervisorState) -> dict:
    """Researcher agent gathers information about the topic."""
    sys = (
        "You are a researcher. Your job is to gather key facts and information "
        "about the given topic. Provide 2-3 key points. Be concise."
    )
    messages_for_llm = [
        ("system", sys),
        ("user", f"Research this topic: {state['topic']}")
    ]
    resp = llm.invoke(messages_for_llm)
    research_msg = f"RESEARCHER: {resp.content}"
    return {"messages": [research_msg]}


def expert_agent(state: SupervisorState) -> dict:
    """Expert agent analyzes and provides insights based on research."""
    sys = (
        "You are an expert analyst. Review the research provided and give "
        "your expert analysis and conclusions. Be specific and insightful."
    )
    # Get context from previous messages
    context = "\n".join(state["messages"])
    messages_for_llm = [
        ("system", sys),
        ("user", f"Topic: {state['topic']}\n\nPrevious research:\n{context}\n\nProvide your expert analysis.")
    ]
    resp = llm.invoke(messages_for_llm)
    expert_msg = f"EXPERT: {resp.content}"
    return {"messages": [expert_msg]}


def supervisor_agent(state: SupervisorState) -> dict:
    """Supervisor decides which agent should act next or if discussion should end."""
    sys = (
        "You are a supervisor managing a research discussion between a RESEARCHER and an EXPERT. "
        "Based on the conversation so far, decide what should happen next:\n"
        "- Return 'researcher' if we need initial research or more information\n"
        "- Return 'expert' if research is done and we need expert analysis\n"
        "- Return 'end' if both research and expert analysis are complete\n\n"
        "Respond with ONLY one word: researcher, expert, or end"
    )

    context = "\n".join(state["messages"]) if state["messages"] else "No discussion yet"
    messages_for_llm = [
        ("system", sys),
        ("user", f"Topic: {state['topic']}\n\nConversation:\n{context}\n\nWhat's next?")
    ]
    resp = llm.invoke(messages_for_llm)
    next_step = resp.content.strip().lower()

    # Ensure valid response
    if next_step not in ["researcher", "expert", "end"]:
        next_step = "end"

    return {"next_agent": next_step}


def finalize_answer(state: SupervisorState) -> dict:
    """Compile final answer from the discussion."""
    sys = (
        "Summarize the research discussion into a clear, concise final answer. "
        "Include key findings and expert insights."
    )
    context = "\n".join(state["messages"])
    messages_for_llm = [
        ("system", sys),
        ("user", f"Topic: {state['topic']}\n\nDiscussion:\n{context}\n\nProvide final summary:")
    ]
    resp = llm.invoke(messages_for_llm)
    return {"final_answer": resp.content}


def route_supervisor(state: SupervisorState) -> str:
    """Route based on supervisor's decision."""
    next_agent = state.get("next_agent", "researcher")
    if next_agent == "end":
        return "finalize"
    return next_agent

supervisor_graph = StateGraph(SupervisorState)

supervisor_graph.add_node("supervisor", supervisor_agent)
supervisor_graph.add_node("researcher", researcher_agent)
supervisor_graph.add_node("expert", expert_agent)
supervisor_graph.add_node("finalize", finalize_answer)

supervisor_graph.add_edge(START, "supervisor")

supervisor_graph.add_conditional_edges(
    "supervisor",
    route_supervisor,
    {
        "researcher": "researcher",
        "expert": "expert",
        "finalize": "finalize"
    }
)

supervisor_graph.add_edge("researcher", "supervisor")
supervisor_graph.add_edge("expert", "supervisor")

supervisor_graph.add_edge("finalize", END)

supervisor_agent_graph = supervisor_graph.compile(debug=True)

topic = "What are the main benefits of using LangGraph for building AI agents?"

initial_state = {
    "topic": topic,
    "messages": [],
    "next_agent": "",
    "final_answer": ""
}

result = supervisor_agent_graph.invoke(initial_state)

print(f"TOPIC: {topic}\n")
print("=" * 80)
print("\nDISCUSSION:")
print("-" * 80)
for msg in result["messages"]:
    print(f"\n{msg}\n")
print("=" * 80)
print(f"\nFINAL ANSWER:\n{result['final_answer']}")

[values] {'topic': 'What are the main benefits of using LangGraph for building AI agents?', 'messages': [], 'next_agent': '', 'final_answer': ''}
[updates] {'supervisor': {'next_agent': 'researcher'}}
[values] {'topic': 'What are the main benefits of using LangGraph for building AI agents?', 'messages': [], 'next_agent': 'researcher', 'final_answer': ''}
[updates] {'researcher': {'messages': ['RESEARCHER: 1. **Modular Design**: LangGraph offers a modular architecture that allows developers to easily integrate various components and functionalities, facilitating the rapid development and customization of AI agents.\n\n2. **Enhanced Natural Language Processing**: It leverages advanced natural language processing capabilities, enabling AI agents to understand and generate human-like responses, improving user interaction and engagement.\n\n3. **Scalability and Flexibility**: LangGraph is designed to be scalable, allowing developers to build AI agents that can handle varying workloads and a

In [7]:
%pip install langgraph_swarm

  Using cached langgraph_swarm-0.1.0-py3-none-any.whl.metadata (10 kB)
Using cached langgraph_swarm-0.1.0-py3-none-any.whl (10 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\languages\Python311\python.exe -m pip install --upgrade pip


### Swarm Pattern

In [8]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph_swarm import create_handoff_tool, create_swarm

llm = make_llm()

@tool
def search_flights(origin: str, destination: str) -> str:
    """Search for flights between two cities."""
    return f"Found 3 flights from {origin} to {destination}, cheapest €149."

@tool
def search_hotels(city: str) -> str:
    """Search for hotels in a city."""
    return f"Found 5 hotels in {city}, from €80/night."

# Each agent carries a handoff tool pointing at the other agent by name
flight_agent = create_agent(
    llm,
    tools=[search_flights, create_handoff_tool(
        agent_name="hotel_agent",
        description="Transfer to the hotel agent for accommodation questions.",
    )],
    system_prompt=("You are a flight booking specialist. Handle flights only. "
                   "For hotel or accommodation questions, hand off to hotel_agent."),
    name="flight_agent",          # <-- name is required; handoffs target it
)

hotel_agent = create_agent(
    llm,
    tools=[search_hotels, create_handoff_tool(
        agent_name="flight_agent",
        description="Transfer to the flight agent for flight questions.",
    )],
    system_prompt=("You are a hotel booking specialist. Handle hotels only. "
                   "For flight questions, hand off to flight_agent."),
    name="hotel_agent",
)

# Checkpointer is REQUIRED so the swarm remembers the active agent across turns
swarm = create_swarm(
    [flight_agent, hotel_agent],
    default_active_agent="flight_agent",
).compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "trip-1"}}

# Turn 1 : starts on the flight agent (the default active agent)
r1 = swarm.invoke(
    {"messages": [{"role": "user", "content": "Find a flight from Amsterdam to Rome."}]},
    config,
)
print(r1["messages"][-1].content)

# Turn 2 : same thread. Asking about hotels makes the flight agent hand off to
# the hotel agent, which then stays active for the rest of the conversation.
r2 = swarm.invoke(
    {"messages": [{"role": "user", "content": "Now find me a hotel in Rome."}]},
    config,
)
print(r2["messages"][-1].content)

I found 3 flights from Amsterdam to Rome, with the cheapest ticket priced at €149. If you need more details or want to book a specific flight, let me know!
I found 5 hotels in Rome, with prices starting from €80 per night. If you need more information about the hotels or want to make a booking, just let me know!
